# cli

> Command-line interface for kosha: sync, search, serve, report, install, ignore.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, sys, os, random, string
from fastcore.all import call_parse, Param, Path, L
from kosha.core import Kosha, mv_skill_md, repo_root


## sync

In [ ]:
#| export
@call_parse
def sync(pkgs:str='',     # comma-sep package names, empty=all pyproject deps
         dir:str='',      # repo root to index; empty=auto-detect
         verbose:bool=True):
	"Index repo + packages + call graph. Incremental — safe to re-run."
	k = Kosha(dir=dir or None)
	k.sync(pkgs=pkgs.split(',') if pkgs else None, dir=dir or None, verbose=verbose)


## search

In [ ]:
#| export
@call_parse
def search(q:str,                  # query with optional key:value filters
           limit:int=10,           # max results
           rerank:str='none',      # 'pagerank' to boost by structural centrality
           graph:bool=True,        # enrich with call-graph data
           json_out:bool=False):   # output as JSON lines
	"Search the kosha index. Prints results to stdout."
	k = Kosha()
	results = k.context(q, limit=limit, graph=graph, rerank=rerank)
	for r in results:
		m = r.get('metadata') or {}
		if json_out: print(json.dumps(r, default=str))
		else: print(f"{m.get('mod_name','?')}  pagerank={r.get('pagerank',0):.5f}\n  {str(r.get('content',''))[:120]}\n")


## ni

In [ ]:
#| export
@call_parse
def ni(node:str):  # fully-qualified node name
	"Print node info: meta + callers + callees + co_dispatched as JSON."
	k = Kosha()
	print(json.dumps(dict(k.ni(node)), default=str))


## path

In [ ]:
#| export
@call_parse
def path(a:str, b:str):  # source and target node names
	"Print shortest call path between two nodes as JSON array."
	k = Kosha()
	print(json.dumps(list(k.short_path(a, b))))


## report

In [ ]:
#| export
@call_parse
def report(out:str='KOSHA_REPORT.md'):  # output file path
	"Generate KOSHA_REPORT.md — structural analysis, no LLMs required."
	from kosha.report import report_md
	k = Kosha()
	p = Path(out)
	p.write_text(report_md(k.graph))
	print(f'wrote {p.resolve()}')


## install

In [ ]:
#| export
@call_parse
def install(harness:str='claude'):  # claude|codex|aider|copilot
	"Install SKILL.md project-local and optionally into a global harness location."
	_locs = {'claude': Path.home()/'.claude/skills/kosha/SKILL.md',
	          'codex':  Path.home()/'.codex/skills/kosha/SKILL.md',
	          'aider':  Path.home()/'.aider/skills/kosha/SKILL.md',
	          'copilot': Path(repo_root() or '.')/'.agents/skills/kosha/SKILL.md'}
	Kosha(install_skill=True)  # project-local
	if harness in _locs:
		dest = _locs[harness]; dest.parent.mkdir(parents=True, exist_ok=True)
		src = Path(__file__).parent/'SKILL.md'
		if src.exists(): dest.write_text(src.read_text(encoding='utf-8')); print(f'installed to {dest}')
	else: print(f'unknown harness {harness!r}, wrote project-local only')


## serve

A lightweight stdin/stdout kernel. The `k` object is pre-loaded; each block (terminated by the delimiter) is eval'd/exec'd and the result printed as JSON.

In [ ]:
#| export
def _serve_loop(k, delim):
	"stdin/stdout kernel: read Python until delimiter, exec in k's namespace, print result."
	ns = {'k': k, 'print': print, 'json': json}
	print(f'kosha ready\n{delim}', flush=True)
	buf = []
	for line in sys.stdin:
		if line.rstrip() == delim:
			code = '\n'.join(buf); buf = []
			try:
				res = eval(compile(code, '<kosha>', 'eval'), ns)
				print(json.dumps(res, default=str))
			except SyntaxError:
				try: exec(code, ns); print('ok')
				except Exception as e: print(json.dumps({'error': str(e)}))
			except Exception as e: print(json.dumps({'error': str(e)}))
			print(delim, flush=True)
		else: buf.append(line.rstrip())


In [ ]:
#| export
@call_parse
def serve(backend:str=''):  # 'clikernel' to delegate to clikernel
	"Start a persistent stdin/stdout kernel. `k` is pre-loaded; send Python, get JSON."
	k = Kosha()
	delim = '--' + ''.join(random.choices(string.ascii_lowercase, k=6))
	if backend == 'clikernel':
		try:
			import clikernel; clikernel.serve(bootstrap=f'from kosha import Kosha; k=Kosha()', delim=delim); return
		except ImportError: pass
	_serve_loop(k, delim)


## ignore

In [ ]:
#| export
@call_parse
def ignore(action:str,      # 'add' to append a pattern
           pattern:str=''):  # glob pattern
	"Manage .koshaignore at repo root. Usage: kosha ignore add <pattern>"
	if action != 'add' or not pattern: print('usage: kosha ignore add <pattern>'); return
	p = Path(repo_root() or '.') / '.koshaignore'
	with p.open('a') as f: f.write(f'\n{pattern}')
	print(f'added {pattern!r} to {p}')


## main entry point

In [ ]:
#| export
def main():
	"CLI entry point: dispatch to kosha subcommands."
	cmds = {'sync': sync, 'search': search, 'ni': ni, 'path': path,
	         'report': report, 'install': install, 'serve': serve, 'ignore': ignore}
	if len(sys.argv) < 2 or sys.argv[1] not in cmds:
		print(f'usage: kosha <cmd> [args]\ncmds: {", ".join(cmds)}'); return
	cmd = sys.argv[1]; del sys.argv[1]
	cmds[cmd]()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()